# 2. Model Training & Comparison

Train and compare 5 different machine learning models for loan default prediction.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

## 2.1 Load & Preprocess Data

In [ ]:
# Load data
df = pd.read_csv('../data/loans.csv')
X = df.drop(['LoanID', 'Default'], axis=1)
y = df['Default']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

# Encode categorical variables
categorical_cols = X.select_dtypes(include=['object']).columns
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le

print(f"\nEncoded {len(categorical_cols)} categorical columns")

## 2.2 Train-Test Split

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]:,} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Test set: {X_test.shape[0]:,} samples ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\nClass distribution maintained:")
print(f"Train - No Default: {(y_train==0).sum():,} | Default: {(y_train==1).sum():,}")
print(f"Test - No Default: {(y_test==0).sum():,} | Default: {(y_test==1).sum():,}")

## 2.3 Train Models

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=15),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, max_depth=15),
    'Naive Bayes': GaussianNB(),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42, max_depth=6)
}

# Train models
trained_models = {}
results = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    trained_models[name] = model
    
    # Predictions
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Metrics
    results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1-Score': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_pred_proba)
    }
    print(f"  ✓ ROC-AUC: {results[name]['ROC-AUC']:.4f}")

print("\n✅ All models trained!")

## 2.4 Model Comparison

In [ ]:
# Results DataFrame
results_df = pd.DataFrame(results).T
print(results_df.round(4))

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
results_df['Accuracy'].sort_values().plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Accuracy Comparison', fontweight='bold')
axes[0].set_xlabel('Accuracy')

# ROC-AUC
results_df['ROC-AUC'].sort_values().plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('ROC-AUC Comparison', fontweight='bold')
axes[1].set_xlabel('ROC-AUC')

plt.tight_layout()
plt.show()

print(f"\n🏆 Best Model by ROC-AUC: {results_df['ROC-AUC'].idxmax()} ({results_df['ROC-AUC'].max():.4f})")

## 2.5 Detailed Analysis of Best Model

In [ ]:
# Get best model
best_model_name = results_df['ROC-AUC'].idxmax()
best_model = trained_models[best_model_name]

print(f"Best Model: {best_model_name}")
print(f"\nMetrics:")
for metric, value in results[best_model_name].items():
    print(f"  {metric}: {value:.4f}")

# Confusion matrix
y_pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Default', 'Default'],
            yticklabels=['No Default', 'Default'])
plt.title(f'{best_model_name} - Confusion Matrix', fontweight='bold')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

## 2.6 Save Models

In [ ]:
import joblib

# Save all models
for name, model in trained_models.items():
    filename = f"../models/{name.lower().replace(' ', '_')}_model.pkl"
    joblib.dump(model, filename)
    print(f"✓ Saved {name} to {filename}")

# Save label encoders
joblib.dump(label_encoders, '../models/label_encoders.pkl')
print(f"\n✓ Saved label encoders")